# Pickup Distance Calibration

Place the robot at the exact position where the arm can pick up a bottle cap.
Run cells in order from top to bottom.

**If camera fails: check ribbon cable, then run terminal command:**
```
gst-launch-1.0 nvarguscamerasrc ! nvvidconv ! jpegenc ! filesink location=/tmp/test.jpg
```

## 1. Cleanup — run this first, every time

In [1]:
import subprocess
import time

# Kill any process holding the camera
subprocess.run(
    ['sh', '-c', 'kill $(lsof /dev/video0 2>/dev/null | awk "NR>1 {print $2}") 2>/dev/null'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Release any existing camera instance
try:
    camera.unobserve_all()
    print('Observers cleared.')
except Exception:
    pass

try:
    camera.stop()
    print('Camera stopped.')
except Exception:
    pass

# Restart nvargus
subprocess.run(
    ['sh', '-c', 'systemctl restart nvargus-daemon'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
print('nvargus restarted.')

time.sleep(3)
print('Cleanup done. Safe to initialize.')

nvargus restarted.
Cleanup done. Safe to initialize.


## 2. Imports

In [2]:
import cv2
import json
import base64
from urllib import request
from jetbot import Camera, bgr8_to_jpeg

print('Libraries loaded.')

Libraries loaded.


## 3. Camera Init

If this fails after cleanup: Kernel → Shutdown All Kernels, reopen, try again.

In [3]:
try:
    camera = Camera.instance(width=224, height=224)
except Exception as e:
    print(f"Could not initialize real camera: {e}")
    print("Falling back to MockCamera...")
    try:
        from tests.mock_camera import MockCamera
        camera = MockCamera.instance(width=224, height=224)
    except Exception as e2:
        print(f"Failed to load MockCamera: {e2}. Creating dummy camera...")
        import numpy as np
        class DummyCamera:
            def __init__(self):
                self.value = np.zeros((224, 224, 3), dtype=np.uint8)
            def observe(self, *args, **kwargs): pass
            def unobserve_all(self, *args, **kwargs): pass
            def stop(self, *args, **kwargs): pass
        camera = DummyCamera()
print('Camera ready.')


Could not initialize real camera: Could not initialize camera.  Please see error trace.
Falling back to MockCamera...
Camera ready.


## 4. Config

In [4]:
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"
CONFIDENCE_THRESHOLD = 0.5

print('Config loaded.')

Config loaded.


## 5. Detection Function

In [5]:
def get_cap_info():
    frame = camera.value.copy()

    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        print("Failed to encode frame")
        return

    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")

    payload = {
        "api_key": API_KEY,
        "inputs": {
            "image": {"type": "base64", "value": image_b64},
            "confidence": CONFIDENCE_THRESHOLD
        }
    }

    data = json.dumps(payload).encode("utf-8")
    req = request.Request(
        url=API_URL,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "python-urllib/3.6"
        },
        method="POST"
    )

    with request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode("utf-8"))

    predictions = result["outputs"][0]["predictions"]["predictions"]

    if predictions:
        print("Found {} cap(s)\n".format(len(predictions)))
        for i, pred in enumerate(predictions):
            print("--- Cap {} ---".format(i+1))
            print("confidence : {:.2f}".format(pred['confidence']))
            print("x (center) : {} px  (image center = 112px)".format(pred['x']))
            print("y (center) : {} px".format(pred['y']))
            print("width      : {} px  <-- use this as PICKUP_THRESHOLD".format(pred['width']))
            print("height     : {} px".format(pred['height']))
            print()
    else:
        print("No cap detected.")
        print("Try lowering CONFIDENCE_THRESHOLD or move the robot closer.")

print('Function ready.')

Function ready.


## 6. Run Samples

1. Place robot at exact pickup position
2. Run single or multiple samples
3. Note the **width** value
4. Run 3-4 times to confirm consistency
5. That width = your PICKUP_THRESHOLD

In [6]:
# Single sample
get_cap_info()

No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.


In [7]:
# Multiple samples
NUM_SAMPLES = 5

for i in range(NUM_SAMPLES):
    print("=== Sample {} ===".format(i+1))
    get_cap_info()
    time.sleep(2)

=== Sample 1 ===
No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.
=== Sample 2 ===
No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.
=== Sample 3 ===
No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.
=== Sample 4 ===
No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.
=== Sample 5 ===
No cap detected.
Try lowering CONFIDENCE_THRESHOLD or move the robot closer.


## 7. Stop Camera — always run this when done

In [8]:
camera.stop()
print('Camera stopped.')

Camera stopped.
